# 09. 의미 기반 핵심 정보 OCR 평가

수작업 `gold/structured`에서 만든 **의미 필드·수치 사실(fact)** 을 정답으로 사용한다. 최종 반복 실험은 10개 카드·50페이지에서 API Luna/Terra를 `detail=original`로 각각 2회 실행한다. 실행 코드는 `09_semantic_repeatability_runner.py`에 두고, 모든 산출물은 `notebooks/data/09_core_numeric_condition_ocr_evaluation/runs/`에 보존한다.

- 숫자·단위·대상·조건이 모두 맞아야 fact exact match다.
- OCR 오탈자를 임의 보정하거나, 정답값을 프롬프트에 제공하지 않는다.
- 표/일반 문장은 출력 형식이 아니라 원문 근거이며, 평가는 의미 JSON으로 한다.


In [1]:
from __future__ import annotations

import csv
import json
import os
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI

PROJECT_ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / 'data/ocr_benchmark/gold/structured').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('프로젝트 루트를 찾지 못했습니다.')

GOLD_ROOT = PROJECT_ROOT / 'data/ocr_benchmark/gold/structured'
FACT_GOLD_PATH = PROJECT_ROOT / 'data/ocr_benchmark/gold/critical_rules/critical_rules_v1.json'
VISION_ROOT = PROJECT_ROOT / 'data/ocr_benchmark/vision/vision_raw_text'
UPSTAGE_ROOT = PROJECT_ROOT / 'data/ocr_benchmark/normalized/upstage'
OUTPUT_ROOT = PROJECT_ROOT / 'notebooks/data/09_core_numeric_condition_ocr_evaluation'
PREDICTION_ROOT = OUTPUT_ROOT / 'predictions'
FIELD_EXTRACTION_MODEL = os.getenv('FIELD_EXTRACTION_MODEL', 'gpt-5.4-mini')

load_dotenv(PROJECT_ROOT / '.env')
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('OPENAI_API_KEY가 .env에 없습니다.')

def read_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))

def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

def build_semantic_gold() -> dict[str, Any]:
    cards = []
    for path in sorted(GOLD_ROOT.glob('*/*.json')):
        data = read_json(path)
        facts = []
        for item in data.get('field_labels', []):
            if not item.get('critical') or item.get('id') == 'card_name':
                continue
            facts.append({
                'fact_id': f"field__{item['id']}", 'fact_kind': 'semantic_field',
                'field_id': item['id'], 'page_num': item.get('page_num'),
                'context_terms': item.get('context_terms', []), 'expected': item.get('value'),
                'source': {'structured_field_id': item['id']},
            })
        for item in data.get('numeric_labels', []):
            if not item.get('critical'):
                continue
            expected_numeric = {'value': item.get('normalized_value')}
            if item.get('unit') is not None:
                expected_numeric['unit'] = item['unit']
            facts.append({
                'fact_id': f"numeric__{item['id']}", 'fact_kind': 'numeric_fact',
                'field_id': item['id'], 'page_num': item.get('page_num'),
                'context_terms': item.get('context_terms', []),
                'expected': expected_numeric,
                'source': {'structured_numeric_id': item['id'], 'surface_text': item.get('surface_text')},
            })
        cards.append({'issuer': data['issuer'], 'card_name': data['card_name'], 'source_file': str(path.relative_to(PROJECT_ROOT)), 'facts': facts})
    return {
        'schema_version': 'semantic_fact_gold_v1',
        'created_at': datetime.now(timezone.utc).isoformat(),
        'annotation_scope': '기존 수작업 structured 정답셋의 critical field_labels와 numeric_labels를 의미 사실 단위로 재표현. 값은 수작업 정답셋에서만 사용하며 OCR 구조화 프롬프트에 제공하지 않는다.',
        'cards': cards,
    }

semantic_gold = build_semantic_gold()
write_json(FACT_GOLD_PATH, semantic_gold)
total_facts = sum(len(card['facts']) for card in semantic_gold['cards'])
print(f"정답 사실셋 생성: {len(semantic_gold['cards'])}개 카드, {total_facts}개 critical fact")


정답 사실셋 생성: 10개 카드, 153개 critical fact


In [2]:
def nullable(schema: dict[str, Any]) -> dict[str, Any]:
    return {'anyOf': [schema, {'type': 'null'}]}

def schema_for_value(value: Any) -> dict[str, Any]:
    if isinstance(value, bool):
        return {'type': 'boolean'}
    if isinstance(value, int) and not isinstance(value, bool):
        return {'type': 'integer'}
    if isinstance(value, float):
        return {'type': 'number'}
    if isinstance(value, str) or value is None:
        return {'type': 'string'}
    if isinstance(value, list):
        item_schema = schema_for_value(value[0]) if value else {'type': 'string'}
        return {'type': 'array', 'items': nullable(item_schema)}
    if isinstance(value, dict):
        return {'type': 'object', 'properties': {key: nullable(schema_for_value(item)) for key, item in value.items()}, 'required': list(value), 'additionalProperties': False}
    raise TypeError(f'지원하지 않는 정답 값 타입: {type(value)!r}')

def page_map_from_vision(path: Path) -> dict[int, str]:
    text = path.read_text(encoding='utf-8')
    chunks = re.split(r'(?=^\[PAGE\s+\d+\])', text, flags=re.MULTILINE)
    pages = {}
    for chunk in chunks:
        match = re.match(r'^\[PAGE\s+(\d+)\]\s*', chunk)
        if match:
            pages[int(match.group(1))] = chunk
    return pages

def page_map_from_upstage(path: Path) -> dict[int, str]:
    pages = {}
    for page in read_json(path).get('pages', []):
        parts = [block.get('text', '') for block in page.get('blocks', [])]
        parts += [table.get('content', '') for table in page.get('tables', [])]
        pages[int(page['page_num'])] = '\n'.join(part for part in parts if part)
    return pages

def source_text(engine: str, card: dict[str, Any]) -> str:
    path = (VISION_ROOT if engine == 'vision' else UPSTAGE_ROOT) / card['issuer'] / f"{card['card_name']}{'.txt' if engine == 'vision' else '.json'}"
    pages = page_map_from_vision(path) if engine == 'vision' else page_map_from_upstage(path)
    selected = sorted({fact['page_num'] for fact in card['facts'] if isinstance(fact.get('page_num'), int)})
    missing = [page for page in selected if page not in pages]
    if missing:
        raise ValueError(f'{engine} {card["issuer"]}/{card["card_name"]}: 필요한 페이지가 없습니다: {missing}')
    return '\n\n'.join(f'[PAGE {page}]\n{pages[page]}' for page in selected)

def response_schema(card: dict[str, Any]) -> dict[str, Any]:
    return {
        'type': 'object',
        'properties': {fact['fact_id']: nullable(schema_for_value(fact['expected'])) for fact in card['facts']},
        'required': [fact['fact_id'] for fact in card['facts']],
        'additionalProperties': False,
    }

def extract_facts(client: OpenAI, engine: str, card: dict[str, Any]) -> dict[str, Any]:
    fact_guide = [{'fact_id': fact['fact_id'], 'field_id': fact['field_id'], 'page_num': fact['page_num'], 'context_terms': fact['context_terms']} for fact in card['facts']]
    prompt = (
        '아래 OCR 원문에서 지정된 카드 혜택 사실만 추출해 JSON으로 반환하세요. '
        'OCR 원문에 명시된 내용만 사용하고 추론, 외부지식, OCR 오탈자 보정은 하지 마세요. '
        '대상·조건·수치·단위 중 원문 근거가 부족하면 그 fact 전체를 null로 반환하세요. '
        '숫자는 스키마 타입에 맞게 의미값으로 정규화하세요(예: 2%는 0.02, 금액은 원 단위 정수). '
        '정답값은 제공되지 않으며, context_terms는 찾을 위치를 위한 힌트일 뿐 값이 아닙니다.\n\n'
        f'카드: {card["issuer"]} / {card["card_name"]}\n'
        f'추출 대상: {json.dumps(fact_guide, ensure_ascii=False)}\n\n'
        f'OCR 원문:\n{source_text(engine, card)}'
    )
    response = client.responses.create(
        model=FIELD_EXTRACTION_MODEL,
        input=prompt,
        text={'format': {'type': 'json_schema', 'name': 'semantic_ocr_fact_prediction', 'strict': True, 'schema': response_schema(card)}},
        store=False,
    )
    return json.loads(response.output_text)

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
run_rows, errors = [], []
for engine in ('vision', 'upstage'):
    for card in semantic_gold['cards']:
        destination = PREDICTION_ROOT / engine / card['issuer'] / f"{card['card_name']}.json"
        try:
            prediction = extract_facts(client, engine, card)
            write_json(destination, {'engine': engine, 'model': FIELD_EXTRACTION_MODEL, 'issuer': card['issuer'], 'card_name': card['card_name'], 'predictions': prediction})
            run_rows.append({'engine': engine, 'issuer': card['issuer'], 'card_name': card['card_name'], 'status': 'success', 'path': str(destination.relative_to(PROJECT_ROOT))})
            print(f"완료: {engine:7} {card['issuer']}/{card['card_name']}")
        except Exception as exc:
            errors.append({'engine': engine, 'issuer': card['issuer'], 'card_name': card['card_name'], 'error_type': type(exc).__name__, 'error': str(exc)})
            run_rows.append({'engine': engine, 'issuer': card['issuer'], 'card_name': card['card_name'], 'status': 'failed', 'path': ''})
            print(f"실패: {engine:7} {card['issuer']}/{card['card_name']} - {type(exc).__name__}: {exc}")
write_json(OUTPUT_ROOT / 'prediction_errors.json', errors)
print(f"구조화 완료: 성공 {sum(row['status'] == 'success' for row in run_rows)}/20, 실패 {len(errors)}")


완료: vision  BC/BC_Biz_AirMoney


완료: vision  NH/NH_Namu_NH


완료: vision  hana/Hana_One_More_SOHO


완료: vision  hyundai/Hyundai_The_Orange_20260330


완료: vision  ibk/IBK_Point3.8(Credit)


...

완료: upstage kookmin/Kookmin_Friend_20210917


완료: upstage lotte/Lotte_LOCA_LIKIT_Eat


완료: upstage samsung/Samsung_iD_ALL


완료: upstage shinhan/Shinhan_Toss_Mr.Life_20251231


완료: upstage woori/Woori_Classic_EVERY_MILE_SKYPASS
구조화 완료: 성공 20/20, 실패 0


In [3]:
def normalized_text(value: str) -> str:
    return re.sub(r'\s+', '', value).casefold()

def values_equal(expected: Any, actual: Any) -> bool:
    if type(expected) in (int, float) and type(actual) in (int, float):
        return abs(float(expected) - float(actual)) < 1e-12
    if isinstance(expected, str) and isinstance(actual, str):
        return normalized_text(expected) == normalized_text(actual)
    if isinstance(expected, list) and isinstance(actual, list):
        remaining = list(actual)
        for item in expected:
            match = next((index for index, candidate in enumerate(remaining) if values_equal(item, candidate)), None)
            if match is None:
                return False
            remaining.pop(match)
        return not remaining
    if isinstance(expected, dict) and isinstance(actual, dict):
        return set(expected) == set(actual) and all(values_equal(expected[key], actual[key]) for key in expected)
    return expected == actual

def align_actual(expected: Any, actual: Any) -> Any:
    if isinstance(expected, dict) and isinstance(actual, dict):
        return {key: align_actual(value, actual.get(key)) for key, value in expected.items()}
    return actual

def leaves(value: Any, kind: str = '') -> list[tuple[str, Any]]:
    if isinstance(value, dict):
        return [leaf for key, item in value.items() for leaf in leaves(item, f'{kind}.{key}'.strip('.'))]
    if isinstance(value, list):
        return [leaf for index, item in enumerate(value) for leaf in leaves(item, f'{kind}[{index}]')]
    return [(kind, value)]

details = []
for engine in ('vision', 'upstage'):
    for card in semantic_gold['cards']:
        path = PREDICTION_ROOT / engine / card['issuer'] / f"{card['card_name']}.json"
        if not path.exists():
            continue
        predictions = read_json(path)['predictions']
        for fact in card['facts']:
            expected, actual = fact['expected'], align_actual(fact['expected'], predictions.get(fact['fact_id']))
            numeric = [(name, value) for name, value in leaves(expected) if isinstance(value, (int, float)) and not isinstance(value, bool)]
            strings = [(name, value) for name, value in leaves(expected) if isinstance(value, str)]
            actual_by_name = dict(leaves(actual)) if isinstance(actual, (dict, list)) else {}
            numeric_ok = sum(values_equal(value, actual_by_name.get(name)) for name, value in numeric)
            string_ok = sum(values_equal(value, actual_by_name.get(name)) for name, value in strings)
            unit_expected = expected.get('unit') if isinstance(expected, dict) and 'unit' in expected else None
            unit_actual = actual.get('unit') if isinstance(actual, dict) else None
            details.append({
                'engine': engine, 'issuer': card['issuer'], 'card_name': card['card_name'],
                'fact_id': fact['fact_id'], 'fact_kind': fact['fact_kind'], 'page_num': fact['page_num'],
                'status': 'null_prediction' if actual is None else ('matched' if values_equal(expected, actual) else 'mismatched'),
                'fact_exact': int(values_equal(expected, actual)),
                'numeric_leaves': len(numeric), 'numeric_correct': numeric_ok,
                'string_leaves': len(strings), 'string_correct': string_ok,
                'unit_available': int(unit_expected is not None), 'unit_correct': int(unit_expected is not None and values_equal(unit_expected, unit_actual)),
                'expected': json.dumps(expected, ensure_ascii=False), 'actual': json.dumps(actual, ensure_ascii=False),
            })

fields = list(details[0]) if details else []
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
with (OUTPUT_ROOT / 'semantic_fact_details.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader(); writer.writerows(details)

summary = []
for engine in ('vision', 'upstage'):
    for fact_kind in ('semantic_field', 'numeric_fact', 'all'):
        rows = [row for row in details if row['engine'] == engine and (fact_kind == 'all' or row['fact_kind'] == fact_kind)]
        if not rows:
            continue
        total = len(rows)
        summary.append({
            'engine': engine, 'fact_kind': fact_kind, 'facts': total,
            'fact_exact_match_rate': round(sum(row['fact_exact'] for row in rows) / total, 6),
            'null_prediction_rate': round(sum(row['status'] == 'null_prediction' for row in rows) / total, 6),
            'numeric_leaf_accuracy': round(sum(row['numeric_correct'] for row in rows) / max(1, sum(row['numeric_leaves'] for row in rows)), 6),
            'string_leaf_accuracy': round(sum(row['string_correct'] for row in rows) / max(1, sum(row['string_leaves'] for row in rows)), 6),
            'unit_accuracy': round(sum(row['unit_correct'] for row in rows) / max(1, sum(row['unit_available'] for row in rows)), 6),
        })
write_json(OUTPUT_ROOT / 'semantic_fact_summary.json', {'model': FIELD_EXTRACTION_MODEL, 'generated_at': datetime.now(timezone.utc).isoformat(), 'summary': summary, 'errors': errors})
print(json.dumps(summary, ensure_ascii=False, indent=2))
print(f"상세 결과: {(OUTPUT_ROOT / 'semantic_fact_details.csv').relative_to(PROJECT_ROOT)}")


[
  {
    "engine": "vision",
    "fact_kind": "semantic_field",
    "facts": 66,
    "fact_exact_match_rate": 0.651515,
    "null_prediction_rate": 0.0,
    "numeric_leaf_accuracy": 0.983516,
    "string_leaf_accuracy": 0.658824,
    "unit_accuracy": 0.0
  },
  {
    "engine": "vision",
    "fact_kind": "numeric_fact",
    "facts": 87,
    "fact_exact_match_rate": 0.494253,
    "null_prediction_rate": 0.011494,
    "numeric_leaf_accuracy": 0.988636,
    "string_leaf_accuracy": 0.47619,
    "unit_accuracy": 0.47619
  },
  {
    "engine": "vision",
    "fact_kind": "all",
    "facts": 153,
    "fact_exact_match_rate": 0.562092,
    "null_prediction_rate": 0.006536,
    "numeric_leaf_accuracy": 0.985185,
    "string_leaf_accuracy": 0.598425,
    "unit_accuracy": 0.465116
  },
  {
    "engine": "upstage",
    "fact_kind": "semantic_field",
    "facts": 66,
    "fact_exact_match_rate": 0.530303,
    "null_prediction_rate": 0.0,
    "numeric_leaf_accuracy": 0.945055,
    "string_leaf_accura